In [33]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#  Obtener la ruta del proyecto
archivo_parquet = "data/16nov25/msci_world.parquet"

# Cargar datos
df = pd.read_parquet(archivo_parquet, engine="pyarrow")
print(f"Datos cargados desde: {archivo_parquet}")
print(f"Shape del DataFrame: {df.shape}")

# Calcular returns diarios (porcentaje)
df["Returns"] = df["Close"].pct_change() * 100

# Calcular returns logarítmicos
df["Log_Returns"] = np.log(df["Close"] / df["Close"].shift(1)) * 100

# Calcular volatilidad móvil (rolling 30 días)
df["Volatility_30d"] = df["Returns"].rolling(window=30).std()

# Calcular media móvil de 50 y 200 días
df["MA_50"] = df["Close"].rolling(window=50).mean()
df["MA_200"] = df["Close"].rolling(window=200).mean()

print("Métricas calculadas:")
print("- Returns (diarios, %): Calculado")
print("- Log Returns (diarios, %): Calculado")
print("- Volatilidad móvil (30 días): Calculado")
print("- Media móvil 50 días: Calculado")
print("- Media móvil 200 días: Calculado")


Datos cargados desde: data/16nov25/msci_world.parquet
Shape del DataFrame: (3490, 8)
Métricas calculadas:
- Returns (diarios, %): Calculado
- Log Returns (diarios, %): Calculado
- Volatilidad móvil (30 días): Calculado
- Media móvil 50 días: Calculado
- Media móvil 200 días: Calculado


## 7. Análisis de Condiciones para EMD (Empirical Mode Decomposition)

Este análisis evalúa si la serie temporal cumple las condiciones ideales para la descomposición en IMFs (Intrinsic Mode Functions) mediante EMD. Las condiciones clave son:

1. **No estacionaridad**: EMD funciona mejor con series no estacionarias
2. **No linealidad**: EMD es adecuado para procesos no lineales
3. **Amplitud y frecuencia variables**: Las IMFs capturan modos con diferentes escalas temporales
4. **Condiciones de Hilbert-Huang**: La serie debe cumplir ciertas condiciones para la transformada de Hilbert


In [34]:
# Importar librerías para análisis estadístico y matemático
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
from scipy.signal import hilbert, find_peaks
from scipy.stats import jarque_bera, normaltest
import warnings
warnings.filterwarnings('ignore')

# Para análisis de no linealidad
try:
    from statsmodels.tsa.stattools import bds
    BDS_AVAILABLE = True
except ImportError:
    BDS_AVAILABLE = False
    print("BDS test no disponible. Instalar statsmodels con versión completa.")

print("Librerías importadas correctamente")


Librerías importadas correctamente


### 8.1. Análisis de Estacionaridad

La estacionaridad es una propiedad clave para EMD. EMD funciona mejor con series **no estacionarias** ya que puede capturar tendencias y componentes de diferentes escalas temporales.


In [35]:
def realizar_tests_estacionaridad(serie: pd.Series, nombre: str = "Serie") -> dict:
    """
    Realiza múltiples tests de estacionaridad sobre una serie temporal.
    
    Parameters
    ----------
    serie : pd.Series
        Serie temporal a analizar.
    nombre : str
        Nombre descriptivo de la serie.
        
    Returns
    -------
    dict
        Diccionario con los resultados de los tests.
    """
    resultados = {}
    
    # Eliminar valores NaN
    serie_limpia = serie.dropna()
    
    # 1. Test de Dickey-Fuller Aumentado (ADF)
    # H0: La serie tiene una raíz unitaria (no estacionaria)
    # H1: La serie es estacionaria
    try:
        resultado_adf = adfuller(serie_limpia, autolag='AIC')
        estadistico = resultado_adf[0]
        p_valor = resultado_adf[1]
        # Acceder a valores_criticos de forma segura
        if len(resultado_adf) > 4:
            valores_criticos = resultado_adf[4]
        else:
            valores_criticos = {}
        resultados['ADF'] = {
            'estadistico': estadistico,
            'p_valor': p_valor,
            'valores_criticos': valores_criticos,
            'es_estacionaria': p_valor < 0.05,
            'interpretacion': 'Estacionaria' if p_valor < 0.05 else 'No estacionaria'
        }
    except Exception as e:
        resultados['ADF'] = {'error': str(e)}
    
    # 2. Test KPSS
    # H0: La serie es estacionaria (tendencia estacionaria)
    # H1: La serie tiene raíz unitaria
    try:
        resultado_kpss = kpss(serie_limpia, regression='ct', nlags='auto')
        resultados['KPSS'] = {
            'estadistico': resultado_kpss[0],
            'p_valor': resultado_kpss[1],
            'valores_criticos': resultado_kpss[3],
            'es_estacionaria': resultado_kpss[1] > 0.05,
            'interpretacion': 'Estacionaria' if resultado_kpss[1] > 0.05 else 'No estacionaria'
        }
    except Exception as e:
        resultados['KPSS'] = {'error': str(e)}
    
    return resultados

# Aplicar tests a la serie de precios de cierre
resultados_estacionaridad_precio = realizar_tests_estacionaridad(
    pd.Series(df['Close']), 
    nombre="Precio de Cierre"
)

# Aplicar tests a los returns (primera diferencia)
resultados_estacionaridad_returns = realizar_tests_estacionaridad(
    df['Returns'].dropna(), 
    nombre="Returns"
)

print("=" * 70)
print("RESULTADOS DE TESTS DE ESTACIONARIDAD")
print("=" * 70)

print("\n1. PRECIO DE CIERRE (Close):")
print("-" * 70)
if 'ADF' in resultados_estacionaridad_precio and 'error' not in resultados_estacionaridad_precio['ADF']:
    print(f"ADF Test:")
    print(f"  Estadístico: {resultados_estacionaridad_precio['ADF']['estadistico']:.4f}")
    print(f"  p-valor: {resultados_estacionaridad_precio['ADF']['p_valor']:.6f}")
    print(f"  Interpretación: {resultados_estacionaridad_precio['ADF']['interpretacion']}")
    print(f"  Valores críticos:")
    for nivel, valor in resultados_estacionaridad_precio['ADF']['valores_criticos'].items():
        print(f"    {nivel}: {valor:.4f}")

if 'KPSS' in resultados_estacionaridad_precio and 'error' not in resultados_estacionaridad_precio['KPSS']:
    print(f"\nKPSS Test:")
    print(f"  Estadístico: {resultados_estacionaridad_precio['KPSS']['estadistico']:.4f}")
    print(f"  p-valor: {resultados_estacionaridad_precio['KPSS']['p_valor']:.6f}")
    print(f"  Interpretación: {resultados_estacionaridad_precio['KPSS']['interpretacion']}")

print("\n2. RETURNS (Primera diferencia):")
print("-" * 70)
if 'ADF' in resultados_estacionaridad_returns and 'error' not in resultados_estacionaridad_returns['ADF']:
    print(f"ADF Test:")
    print(f"  Estadístico: {resultados_estacionaridad_returns['ADF']['estadistico']:.4f}")
    print(f"  p-valor: {resultados_estacionaridad_returns['ADF']['p_valor']:.6f}")
    print(f"  Interpretación: {resultados_estacionaridad_returns['ADF']['interpretacion']}")

if 'KPSS' in resultados_estacionaridad_returns and 'error' not in resultados_estacionaridad_returns['KPSS']:
    print(f"\nKPSS Test:")
    print(f"  Estadístico: {resultados_estacionaridad_returns['KPSS']['estadistico']:.4f}")
    print(f"  p-valor: {resultados_estacionaridad_returns['KPSS']['p_valor']:.6f}")
    print(f"  Interpretación: {resultados_estacionaridad_returns['KPSS']['interpretacion']}")

print("\n" + "=" * 70)
print("CONCLUSIÓN PARA EMD:")
print("=" * 70)
if 'ADF' in resultados_estacionaridad_precio and 'error' not in resultados_estacionaridad_precio['ADF']:
    if not resultados_estacionaridad_precio['ADF']['es_estacionaria']:
        print("✓ La serie de precios es NO ESTACIONARIA - Condición IDEAL para EMD")
    else:
        print("⚠ La serie de precios es ESTACIONARIA - EMD puede funcionar pero con limitaciones")


RESULTADOS DE TESTS DE ESTACIONARIDAD

1. PRECIO DE CIERRE (Close):
----------------------------------------------------------------------
ADF Test:
  Estadístico: 1.0542
  p-valor: 0.994806
  Interpretación: No estacionaria
  Valores críticos:
    1%: -3.4322
    5%: -2.8624
    10%: -2.5672

KPSS Test:
  Estadístico: 1.2110
  p-valor: 0.010000
  Interpretación: No estacionaria

2. RETURNS (Primera diferencia):
----------------------------------------------------------------------
ADF Test:
  Estadístico: -19.5227
  p-valor: 0.000000
  Interpretación: Estacionaria

KPSS Test:
  Estadístico: 0.0341
  p-valor: 0.100000
  Interpretación: Estacionaria

CONCLUSIÓN PARA EMD:
✓ La serie de precios es NO ESTACIONARIA - Condición IDEAL para EMD


### 8.2. Análisis de No Linealidad

EMD es especialmente útil para series no lineales, ya que no requiere supuestos de linealidad como otros métodos (ARIMA, etc.). Verificamos la presencia de no linealidad mediante varios tests.


In [36]:
def analizar_no_linealidad(serie: pd.Series, nombre: str = "Serie") -> dict:
    """
    Analiza la no linealidad de una serie temporal mediante múltiples métodos.
    
    Parameters
    ----------
    serie : pd.Series
        Serie temporal a analizar.
    nombre : str
        Nombre descriptivo de la serie.
        
    Returns
    -------
    dict
        Diccionario con los resultados del análisis.
    """
    resultados = {}
    serie_limpia = serie.dropna()
    
    # 1. Test de BDS (Brock-Dechert-Scheinkman)
    # Detecta dependencia no lineal en los residuos
    if BDS_AVAILABLE:
        try:
            # Usar los residuos de un modelo AR(1) para el test BDS
            from statsmodels.tsa.ar_model import AutoReg
            modelo_ar = AutoReg(serie_limpia, lags=1).fit()
            residuos = modelo_ar.resid
            
            # BDS test con diferentes dimensiones de embedding
            dimensiones = [2, 3, 4, 5]
            resultados_bds = {}
            for dim in dimensiones:
                try:
                    estadistico, p_valor = bds(residuos, max_dim=dim)
                    resultados_bds[dim] = {
                        'estadistico': estadistico[-1] if isinstance(estadistico, (list, np.ndarray)) else estadistico,
                        'p_valor': p_valor[-1] if isinstance(p_valor, (list, np.ndarray)) else p_valor,
                        'es_no_lineal': p_valor[-1] < 0.05 if isinstance(p_valor, (list, np.ndarray)) else p_valor < 0.05
                    }
                except:
                    pass
            
            resultados['BDS'] = resultados_bds
        except Exception as e:
            resultados['BDS'] = {'error': str(e)}
    
    # 2. Análisis de autocorrelación no lineal
    # Calcular autocorrelación de los cuadrados (detecta heterocedasticidad/no linealidad)
    try:
        serie_cuadrada = serie_limpia ** 2
        acf_cuadrados = acf(serie_cuadrada, nlags=20, fft=True)
        resultados['ACF_cuadrados'] = {
            'lags_significativos': sum([abs(x) > 1.96/np.sqrt(len(serie_limpia)) for x in acf_cuadrados[1:]]),
            'max_autocorr': np.max(np.abs(acf_cuadrados[1:])),
            'interpretacion': 'Presencia de dependencia no lineal' if sum([abs(x) > 1.96/np.sqrt(len(serie_limpia)) for x in acf_cuadrados[1:]]) > 0 else 'Sin dependencia no lineal evidente'
        }
    except Exception as e:
        resultados['ACF_cuadrados'] = {'error': str(e)}
    
    # 3. Test de Ljung-Box en los cuadrados (detecta ARCH/GARCH)
    try:
        serie_cuadrada = serie_limpia ** 2
        lb_test = acorr_ljungbox(serie_cuadrada, lags=10, return_df=True)
        resultados['Ljung_Box_cuadrados'] = {
            'estadistico': lb_test['lb_stat'].iloc[-1],
            'p_valor': lb_test['lb_pvalue'].iloc[-1],
            'es_no_lineal': lb_test['lb_pvalue'].iloc[-1] < 0.05,
            'interpretacion': 'Presencia de efectos ARCH/GARCH (no linealidad)' if lb_test['lb_pvalue'].iloc[-1] < 0.05 else 'Sin efectos ARCH/GARCH evidentes'
        }
    except Exception as e:
        resultados['Ljung_Box_cuadrados'] = {'error': str(e)}
    
    # 4. Asimetría y curtosis (indicadores de no normalidad/no linealidad)
    try:
        skew = serie_limpia.skew()
        kurt = serie_limpia.kurtosis()
        resultados['distribucion'] = {
            'skewness': skew,
            'kurtosis': kurt,
            'es_normal': abs(skew) < 0.5 and abs(kurt) < 0.5,
            'interpretacion': f'Distribución {"normal" if abs(skew) < 0.5 and abs(kurt) < 0.5 else "no normal"} (skew={skew:.4f}, kurt={kurt:.4f})'
        }
        
        # Test de Jarque-Bera
        jb_stat, jb_pval = jarque_bera(serie_limpia)
        resultados['Jarque_Bera'] = {
            'estadistico': jb_stat,
            'p_valor': jb_pval,
            'es_normal': jb_pval > 0.05,
            'interpretacion': 'Distribución normal' if jb_pval > 0.05 else 'Distribución no normal'
        }
    except Exception as e:
        resultados['distribucion'] = {'error': str(e)}
    
    return resultados

# Analizar no linealidad en precios y returns
resultados_no_linealidad_precio = analizar_no_linealidad(df['Close'], "Precio de Cierre")
resultados_no_linealidad_returns = analizar_no_linealidad(df['Returns'].dropna(), "Returns")

print("=" * 70)
print("ANÁLISIS DE NO LINEALIDAD")
print("=" * 70)

print("\n1. PRECIO DE CIERRE (Close):")
print("-" * 70)

if 'ACF_cuadrados' in resultados_no_linealidad_precio and 'error' not in resultados_no_linealidad_precio['ACF_cuadrados']:
    print(f"Autocorrelación de Cuadrados:")
    print(f"  Lags significativos: {resultados_no_linealidad_precio['ACF_cuadrados']['lags_significativos']}")
    print(f"  Máxima autocorrelación: {resultados_no_linealidad_precio['ACF_cuadrados']['max_autocorr']:.4f}")
    print(f"  Interpretación: {resultados_no_linealidad_precio['ACF_cuadrados']['interpretacion']}")

if 'Ljung_Box_cuadrados' in resultados_no_linealidad_precio and 'error' not in resultados_no_linealidad_precio['Ljung_Box_cuadrados']:
    print(f"\nTest de Ljung-Box (cuadrados):")
    print(f"  Estadístico: {resultados_no_linealidad_precio['Ljung_Box_cuadrados']['estadistico']:.4f}")
    print(f"  p-valor: {resultados_no_linealidad_precio['Ljung_Box_cuadrados']['p_valor']:.6f}")
    print(f"  Interpretación: {resultados_no_linealidad_precio['Ljung_Box_cuadrados']['interpretacion']}")

if 'distribucion' in resultados_no_linealidad_precio and 'error' not in resultados_no_linealidad_precio['distribucion']:
    print(f"\nAnálisis de Distribución:")
    print(f"  {resultados_no_linealidad_precio['distribucion']['interpretacion']}")
    if 'Jarque_Bera' in resultados_no_linealidad_precio and 'error' not in resultados_no_linealidad_precio['Jarque_Bera']:
        print(f"  Test Jarque-Bera: p-valor = {resultados_no_linealidad_precio['Jarque_Bera']['p_valor']:.6f}")
        print(f"    Interpretación: {resultados_no_linealidad_precio['Jarque_Bera']['interpretacion']}")

if 'BDS' in resultados_no_linealidad_precio and 'error' not in resultados_no_linealidad_precio['BDS']:
    print(f"\nTest BDS (Brock-Dechert-Scheinkman):")
    for dim, res in resultados_no_linealidad_precio['BDS'].items():
        if 'error' not in res:
            print(f"  Dimensión {dim}: estadístico={res['estadistico']:.4f}, p-valor={res['p_valor']:.6f}")
            print(f"    {'No linealidad detectada' if res['es_no_lineal'] else 'Sin evidencia de no linealidad'}")

print("\n2. RETURNS:")
print("-" * 70)
if 'ACF_cuadrados' in resultados_no_linealidad_returns and 'error' not in resultados_no_linealidad_returns['ACF_cuadrados']:
    print(f"Autocorrelación de Cuadrados:")
    print(f"  Lags significativos: {resultados_no_linealidad_returns['ACF_cuadrados']['lags_significativos']}")
    print(f"  Interpretación: {resultados_no_linealidad_returns['ACF_cuadrados']['interpretacion']}")

if 'Ljung_Box_cuadrados' in resultados_no_linealidad_returns and 'error' not in resultados_no_linealidad_returns['Ljung_Box_cuadrados']:
    print(f"\nTest de Ljung-Box (cuadrados):")
    print(f"  p-valor: {resultados_no_linealidad_returns['Ljung_Box_cuadrados']['p_valor']:.6f}")
    print(f"  Interpretación: {resultados_no_linealidad_returns['Ljung_Box_cuadrados']['interpretacion']}")

print("\n" + "=" * 70)
print("CONCLUSIÓN PARA EMD:")
print("=" * 70)
if 'Ljung_Box_cuadrados' in resultados_no_linealidad_precio and 'error' not in resultados_no_linealidad_precio['Ljung_Box_cuadrados']:
    if resultados_no_linealidad_precio['Ljung_Box_cuadrados']['es_no_lineal']:
        print("✓ Se detecta NO LINEALIDAD - Condición IDEAL para EMD")
    else:
        print("⚠ No se detecta no linealidad evidente - EMD puede funcionar pero puede haber métodos lineales más eficientes")


ANÁLISIS DE NO LINEALIDAD

1. PRECIO DE CIERRE (Close):
----------------------------------------------------------------------
Autocorrelación de Cuadrados:
  Lags significativos: 20
  Máxima autocorrelación: 0.9978
  Interpretación: Presencia de dependencia no lineal

Test de Ljung-Box (cuadrados):
  Estadístico: 34210.6937
  p-valor: 0.000000
  Interpretación: Presencia de efectos ARCH/GARCH (no linealidad)

Análisis de Distribución:
  Distribución no normal (skew=0.7515, kurt=-0.3538)
  Test Jarque-Bera: p-valor = 0.000000
    Interpretación: Distribución no normal

Test BDS (Brock-Dechert-Scheinkman):
  Dimensión 3: estadístico=20.9882, p-valor=0.000000
    No linealidad detectada
  Dimensión 4: estadístico=24.9499, p-valor=0.000000
    No linealidad detectada
  Dimensión 5: estadístico=27.8287, p-valor=0.000000
    No linealidad detectada

2. RETURNS:
----------------------------------------------------------------------
Autocorrelación de Cuadrados:
  Lags significativos: 20
  In

In [37]:
def analizar_variabilidad_amplitud_frecuencia(serie: pd.Series, ventana: int = 252) -> tuple[dict, pd.Series | pd.DataFrame, list | None]:
    """
    Analiza la variabilidad en amplitud y frecuencia de una serie temporal.
    
    Parameters
    ----------
    serie : pd.Series
        Serie temporal a analizar.
    ventana : int
        Tamaño de la ventana móvil para el análisis. Se establece por defecto a 252 días de negociación (aproximadamente 1 año).
        
    Returns
    -------
    tuple[dict, pd.Series | pd.DataFrame, list | None]
        Tupla con:
        - Diccionario con los resultados del análisis.
        - Serie o DataFrame con la volatilidad móvil.
        - Lista con las frecuencias móviles o None si no se calculó.
    """
    resultados = {}
    serie_limpia = serie.dropna()
    
    # 1. Variabilidad en amplitud (volatilidad móvil)
    volatilidad_movil = serie_limpia.rolling(window=ventana).std()
    resultados['variabilidad_amplitud'] = {
        'volatilidad_promedio': volatilidad_movil.mean(),
        'volatilidad_std': volatilidad_movil.std(),
        'coeficiente_variacion': volatilidad_movil.std() / volatilidad_movil.mean() if volatilidad_movil.mean() > 0 else 0,
        'ratio_max_min': volatilidad_movil.max() / volatilidad_movil.min() if volatilidad_movil.min() > 0 else 0,
        'interpretacion': 'Alta variabilidad en amplitud' if (volatilidad_movil.std() / volatilidad_movil.mean() > 0.3) else 'Variabilidad moderada en amplitud'
    }
    
    # 2. Análisis de frecuencia mediante conteo de cruces por cero
    # Una medida aproximada de la frecuencia dominante
    cambios_signo = np.diff(np.sign(serie_limpia - serie_limpia.rolling(window=ventana).mean()))
    cruces_cero = np.sum(cambios_signo != 0)
    
    # Frecuencia móvil (cruces por cero en ventanas)
    frecuencias_moviles = []
    for i in range(ventana, len(serie_limpia)):
        ventana_serie = serie_limpia.iloc[i-ventana:i]
        media_ventana = ventana_serie.mean()
        cambios = np.diff(np.sign(ventana_serie - media_ventana))
        frecuencias_moviles.append(np.sum(cambios != 0))
    
    if len(frecuencias_moviles) > 0:
        resultados['variabilidad_frecuencia'] = {
            'cruces_cero_totales': cruces_cero,
            'frecuencia_promedio': np.mean(frecuencias_moviles),
            'frecuencia_std': np.std(frecuencias_moviles),
            'coeficiente_variacion_freq': np.std(frecuencias_moviles) / np.mean(frecuencias_moviles) if np.mean(frecuencias_moviles) > 0 else 0,
            'interpretacion': 'Alta variabilidad en frecuencia' if (np.std(frecuencias_moviles) / np.mean(frecuencias_moviles) > 0.3) else 'Variabilidad moderada en frecuencia'
        }
    
    # 3. Análisis de rango intercuartílico móvil (medida de dispersión)
    iqr_movil = serie_limpia.rolling(window=ventana).quantile(0.75) - serie_limpia.rolling(window=ventana).quantile(0.25)
    resultados['variabilidad_dispersion'] = {
        'iqr_promedio': iqr_movil.mean(),
        'iqr_std': iqr_movil.std(),
        'coeficiente_variacion_iqr': iqr_movil.std() / iqr_movil.mean() if iqr_movil.mean() > 0 else 0
    }
    
    return resultados, volatilidad_movil, frecuencias_moviles if 'variabilidad_frecuencia' in resultados else None

# Analizar variabilidad
resultados_variabilidad, vol_movil, frec_moviles = analizar_variabilidad_amplitud_frecuencia(pd.Series(df['Close']), ventana=252)

print("=" * 70)
print("ANÁLISIS DE VARIABILIDAD EN AMPLITUD Y FRECUENCIA")
print("=" * 70)

if 'variabilidad_amplitud' in resultados_variabilidad:
    print("\n1. VARIABILIDAD EN AMPLITUD (Volatilidad Móvil - 252 días):")
    print("-" * 70)
    print(f"  Volatilidad promedio: {resultados_variabilidad['variabilidad_amplitud']['volatilidad_promedio']:.4f}")
    print(f"  Desviación estándar de volatilidad: {resultados_variabilidad['variabilidad_amplitud']['volatilidad_std']:.4f}")
    print(f"  Coeficiente de variación: {resultados_variabilidad['variabilidad_amplitud']['coeficiente_variacion']:.4f}")
    print(f"  Ratio máximo/mínimo: {resultados_variabilidad['variabilidad_amplitud']['ratio_max_min']:.4f}")
    print(f"  Interpretación: {resultados_variabilidad['variabilidad_amplitud']['interpretacion']}")

if 'variabilidad_frecuencia' in resultados_variabilidad:
    print("\n2. VARIABILIDAD EN FRECUENCIA:")
    print("-" * 70)
    print(f"  Cruces por cero totales: {resultados_variabilidad['variabilidad_frecuencia']['cruces_cero_totales']}")
    print(f"  Frecuencia promedio (cruces por ventana): {resultados_variabilidad['variabilidad_frecuencia']['frecuencia_promedio']:.2f}")
    print(f"  Desviación estándar de frecuencia: {resultados_variabilidad['variabilidad_frecuencia']['frecuencia_std']:.2f}")
    print(f"  Coeficiente de variación: {resultados_variabilidad['variabilidad_frecuencia']['coeficiente_variacion_freq']:.4f}")
    print(f"  Interpretación: {resultados_variabilidad['variabilidad_frecuencia']['interpretacion']}")

if 'variabilidad_dispersion' in resultados_variabilidad:
    print("\n3. VARIABILIDAD EN DISPERSIÓN (IQR Móvil):")
    print("-" * 70)
    print(f"  IQR promedio: {resultados_variabilidad['variabilidad_dispersion']['iqr_promedio']:.4f}")
    print(f"  Desviación estándar del IQR: {resultados_variabilidad['variabilidad_dispersion']['iqr_std']:.4f}")
    print(f"  Coeficiente de variación: {resultados_variabilidad['variabilidad_dispersion']['coeficiente_variacion_iqr']:.4f}")

print("\n" + "=" * 70)
print("CONCLUSIÓN PARA EMD:")
print("=" * 70)
if 'variabilidad_amplitud' in resultados_variabilidad:
    if resultados_variabilidad['variabilidad_amplitud']['coeficiente_variacion'] > 0.3:
        print("✓ Alta variabilidad en amplitud - Condición IDEAL para EMD")
    else:
        print("⚠ Variabilidad moderada en amplitud - EMD puede funcionar")


ANÁLISIS DE VARIABILIDAD EN AMPLITUD Y FRECUENCIA

1. VARIABILIDAD EN AMPLITUD (Volatilidad Móvil - 252 días):
----------------------------------------------------------------------
  Volatilidad promedio: 4.9841
  Desviación estándar de volatilidad: 2.8331
  Coeficiente de variación: 0.5684
  Ratio máximo/mínimo: 8.9890
  Interpretación: Alta variabilidad en amplitud

2. VARIABILIDAD EN FRECUENCIA:
----------------------------------------------------------------------
  Cruces por cero totales: 325
  Frecuencia promedio (cruces por ventana): 12.28
  Desviación estándar de frecuencia: 6.73
  Coeficiente de variación: 0.5479
  Interpretación: Alta variabilidad en frecuencia

3. VARIABILIDAD EN DISPERSIÓN (IQR Móvil):
----------------------------------------------------------------------
  IQR promedio: 7.4015
  Desviación estándar del IQR: 4.6768
  Coeficiente de variación: 0.6319

CONCLUSIÓN PARA EMD:
✓ Alta variabilidad en amplitud - Condición IDEAL para EMD


In [38]:
# Visualización de variabilidad en amplitud y frecuencia
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        "MSCI World close price",
        "Amplitude variability (252 days)",
        "Frequency variability (252 days)"
    ),
    vertical_spacing=0.1,
    row_heights=[0.4, 0.3, 0.3]
)

# Precio con banda de volatilidad
fig.add_trace(
    go.Scatter(x=df.index, y=df["Close"], name="Close", line=dict(color="blue")),
    row=1, col=1
)

# Volatilidad móvil
if vol_movil is not None:
    fig.add_trace(
        go.Scatter(
            x=vol_movil.index, 
            y=vol_movil.values, 
            name="Volatilidad Móvil", 
            line=dict(color="red"),
            fill="tozeroy"
        ),
        row=2, col=1
    )

# Frecuencia móvil
if frec_moviles is not None and len(frec_moviles) > 0:
    indices_frec = df.index[252:252+len(frec_moviles)]
    fig.add_trace(
        go.Scatter(
            x=indices_frec, 
            y=frec_moviles, 
            name="Frecuencia Móvil", 
            line=dict(color="green"),
            fill="tozeroy"
        ),
        row=3, col=1
    )

fig.update_layout(
    title="Análisis de Variabilidad en Amplitud y Frecuencia - MSCI World",
    height=900,
    showlegend=True
)

fig.update_xaxes(title_text="Fecha", row=3, col=1)
fig.update_yaxes(title_text="Precio", row=1, col=1)
fig.update_yaxes(title_text="Volatilidad", row=2, col=1)
fig.update_yaxes(title_text="Cruces por Cero", row=3, col=1)

fig.show()


In [39]:
# Resumen de métricas clave
fecha_min_resumen = df.index.min()
fecha_max_resumen = df.index.max()
fecha_max_precio = df['Close'].idxmax()
fecha_min_precio = df['Close'].idxmin()

# Formatear fechas
fecha_min_str = str(fecha_min_resumen).split()[0] if hasattr(fecha_min_resumen, 'date') else str(fecha_min_resumen)
fecha_max_str = str(fecha_max_resumen).split()[0] if hasattr(fecha_max_resumen, 'date') else str(fecha_max_resumen)
fecha_max_precio_str = str(fecha_max_precio).split()[0] if hasattr(fecha_max_precio, 'date') else str(fecha_max_precio)
fecha_min_precio_str = str(fecha_min_precio).split()[0] if hasattr(fecha_min_precio, 'date') else str(fecha_min_precio)

print("=" * 60)
print("RESUMEN DEL ANÁLISIS - MSCI WORLD")
print("=" * 60)
print(f"\nPeríodo analizado: {fecha_min_str} a {fecha_max_str}")
print(f"Total de observaciones: {len(df):,}")
print(f"\nPrecio de Cierre:")
print(f"  - Inicial: ${df['Close'].iloc[0]:.2f}")
print(f"  - Final: ${df['Close'].iloc[-1]:.2f}")
print(f"  - Máximo: ${df['Close'].max():.2f} (Fecha: {fecha_max_precio_str})")
print(f"  - Mínimo: ${df['Close'].min():.2f} (Fecha: {fecha_min_precio_str})")
print(f"\nReturns:")
print(f"  - Media diaria: {df['Returns'].mean():.4f}%")
print(f"  - Desviación estándar: {df['Returns'].std():.4f}%")
print(f"  - Return total acumulado: {((df['Close'].iloc[-1] / df['Close'].iloc[0]) - 1) * 100:.2f}%")
print(f"\nVolatilidad:")
print(f"  - Volatilidad promedio (30d): {df['Volatility_30d'].mean():.4f}%")
print(f"  - Volatilidad máxima (30d): {df['Volatility_30d'].max():.4f}%")
print("=" * 60)


RESUMEN DEL ANÁLISIS - MSCI WORLD

Período analizado: 2012-01-12 a 2025-11-26
Total de observaciones: 3,490

Precio de Cierre:
  - Inicial: $38.79
  - Final: $184.73
  - Máximo: $186.64 (Fecha: 2025-10-28)
  - Mínimo: $38.04 (Fecha: 2012-06-05)

Returns:
  - Media diaria: 0.0506%
  - Desviación estándar: 1.0780%
  - Return total acumulado: 376.27%

Volatilidad:
  - Volatilidad promedio (30d): 0.9417%
  - Volatilidad máxima (30d): 5.1228%
